<a href="https://colab.research.google.com/github/sunnysavita10/Generative-AI-Indepth-Basic-to-Advance/blob/main/MergerRetriever_and_LongContextReorder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install the Require Libraries

In [2]:
!pip install -qU langchain chromadb huggingface_hub sentence-transformers pypdf openai tiktoken

In [3]:
!pip install -U langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 100.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 71.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.7 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-exporter-otlp-proto-common==1.38.0, but you have opentelemetry-exporter-otlp-proto-common 1.43.0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-proto==1.38.0, but you have opentelemetry-proto 1.43.0 which is incompati

# Let's Load the Data Now...

In [4]:
from langchain_community.document_loaders import PyPDFLoader

/tmp/ipykernel_4148/4175148793.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
loader_harrypotter  = PyPDFLoader("/content/harry_potter_book.pdf")

In [7]:
documnet_harrypotter = loader_harrypotter.load()

In [8]:
print(len(documnet_harrypotter))

6


In [9]:
loader_got = PyPDFLoader("/content/got_book.pdf")

In [10]:
documnet_got = loader_got.load()

In [11]:
print(len(documnet_got))


8


# Let's create the text splitter for chunking

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [13]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=100)

In [14]:
text_harrypotter = text_splitter.split_documents(documnet_harrypotter)

In [15]:
text_got = text_splitter.split_documents(documnet_got)

In [16]:
print(len(text_harrypotter))

69


In [17]:
print(len(text_got))

46


# Load the Embedding Model to Conver the Data into Vector

In [18]:
from langchain_community.embeddings import HuggingFaceEmbeddings, OpenAIEmbeddings,HuggingFaceBgeEmbeddings

In [19]:
HF_TOKEN_REMOVEDembeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

/tmp/ipykernel_4148/4150278297.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  HF_TOKEN_REMOVEDembeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [20]:
HF_TOKEN_REMOVEDbge_embeddings = HuggingFaceBgeEmbeddings(model_name="BAAI/bge-large-en")


/tmp/ipykernel_4148/1884637011.py:1: LangChainDeprecationWarning: The class `HuggingFaceBgeEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  HF_TOKEN_REMOVEDbge_embeddings = HuggingFaceBgeEmbeddings(model_name="BAAI/bge-large-en")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/90.3k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [ ]:
from google.colab import userdata
OPENAI_API_KEY=userdata.get('OPENAI_API_KEY')

In [ ]:
os.environ["OPENAI_API_KEY"]=OPENAI_API_KEY

In [ ]:
openai_embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)

# Now ingest the Data into the Chroma Database

In [23]:
from langchain_community.vectorstores import Chroma
import chromadb

In [24]:
import os
os.getcwd()

'/content'

In [25]:
CURRENT_DIR = os.path.dirname(os.path.abspath("."))

In [26]:
CURRENT_DIR

'/'

In [27]:
DB_DIR = os.path.join(CURRENT_DIR, "/content/db")

In [28]:
DB_DIR

'/content/db'

In [29]:
client_settings = chromadb.config.Settings(
    is_persistent=True,
    persist_directory=DB_DIR,
    anonymized_telemetry=False,
)

In [30]:
harrypotter_vectorstore = Chroma.from_documents(text_harrypotter,
                                       HF_TOKEN_REMOVEDbge_embeddings,
                                       client_settings=client_settings,
                                       collection_name="harrypotter",
                                       collection_metadata={"hnsw":"cosine"},
                                       persist_directory="/store/harrypotter")

In [31]:
got_vectorstore = Chroma.from_documents(text_got,
                                       HF_TOKEN_REMOVEDbge_embeddings,
                                       client_settings=client_settings,
                                       collection_name="got",
                                       collection_metadata={"hnsw":"cosine"},
                                       persist_directory="/store/got")

 # Now Crearte a Retriever

In [32]:
retriever_harrypotter = harrypotter_vectorstore.as_retriever(search_type="mmr",search_kwargs={"k": 5, "include_metadata": True})

In [33]:
retriever_got = got_vectorstore.as_retriever(search_type="mmr",search_kwargs={"k": 5, "include_metadata": True})

# Let's Merge both Retriever

# It is also called lord of retriever(LOTR)

In [34]:
import langchain
import langchain_community

print(langchain.__version__)
print(langchain_community.__version__)

1.3.11
0.4.2


In [35]:
pip install -U langchain-classic

In [36]:

from langchain_classic.retrievers import MergerRetriever

In [37]:
lotr = MergerRetriever(retrievers=[retriever_harrypotter, retriever_got])

In [38]:
for doc in lotr.invoke("Who was Jon Snow?"):
    print(doc.page_content)

ZHOU Wangyue
[a],*
; ZHUO Yue
[a]
[a]
School of Foreign Languages, Zhejiang University of Finance & 
Economics, Hangzhou, China.
* 
Corresponding author.
Supported by Zhejiang Federation of Humanities and Social Science 
Circles: Study on Foreign Children’s Literature Translation After 1979 
(No.2012Z71) and Scientific Research Fund of Zhejiang Provincial 
Education: Study on English Children’s Literature Translation After 
1979 (No. Y201223249).
raiders. Each day had been worse than the day that had come before it. Today was the 
worst of all. A cold wind was blowing out of the north, and it made the trees rustle like 
living things. All day, Will had felt as though something were watching him, something 
cold and implacable that loved him not. Gared had felt it too. Will wanted nothing so 
much as to ride hellbent for the safety of the Wall, but that was not a feeling to share 
with your commander.
Education: Study on English Children’s Literature Translation After 
1979 (No. Y201223

In [39]:
for doc in lotr.invoke("Who is a harry potter?"):
    print(doc.page_content)

DOI: http://dx.doi.org/10.3968/j.sll.1923156320130703.3020
INTRODUCTION
The book series Harry Potter enjoy a global popularity 
in recent years. By now most readers are aware of what 
has come to be called the Harry Potter phenomenon. The 
sudden success of the book has resulted in the translations 
of 67 languages. The book is featured with vivid language 
and the wonderful magic world it presents. And both 
children and adults are fascinated by Harry Potter and his 
unusual stories.
with your commander.
Especially not a commander like this one.
Ser Waymar Royce was the youngest son of an ancient house with too many heirs. He 
was a handsome youth of eighteen, grey-eyed and graceful and slender as a knife. 
Mounted on his huge black destrier, the knight towered above Will and Gared on their 
smaller garrons. He wore black leather boots, black woolen pants, black moleskin gloves, 
and a fine supple coat of gleaming black ringmail over layers of black wool and boiled
Education: Study on

## See this result is too much messy now lets refine it according to the question and overcome the situation of lost in middle

# Now After understanding step by step it create a pipeline for LLM

In [40]:
pip install -U langchain-classic

In [41]:
from langchain_classic.document_transformers import (
    EmbeddingsClusteringFilter,
    EmbeddingsRedundantFilter,
    LongContextReorder,
)

from langchain_classic.retrievers.document_compressors import (
    DocumentCompressorPipeline,
)

from langchain_classic.retrievers import ContextualCompressionRetriever

In [42]:
from re import search
filter = EmbeddingsRedundantFilter(embeddings=HF_TOKEN_REMOVEDbge_embeddings)
reordering = LongContextReorder()
pipeline = DocumentCompressorPipeline(transformers=[filter, reordering])
compression_retriever_reordered = ContextualCompressionRetriever(
    base_compressor=pipeline, base_retriever=lotr,search_kwargs={"k": 3, "include_metadata": True}
)


In [46]:
docs = compression_retriever_reordered.invoke("Who is Ser Waymar ?")

print(len(docs))
print(docs[0].page_content)

10
with your commander.
Especially not a commander like this one.
Ser Waymar Royce was the youngest son of an ancient house with too many heirs. He 
was a handsome youth of eighteen, grey-eyed and graceful and slender as a knife. 
Mounted on his huge black destrier, the knight towered above Will and Gared on their 
smaller garrons. He wore black leather boots, black woolen pants, black moleskin gloves, 
and a fine supple coat of gleaming black ringmail over layers of black wool and boiled


# Load the model from huggingface

In [44]:
!pip install llama-cpp-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 MB 13.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.1 MB/s eta 0:00:00
  Created wheel for llama-cpp-python: filename=llama_cpp_python-0.3.32-py3-none-linux_x86_64.whl size=20122230 sha256=c096f288d9577997a9e1b8362850980ae2750d4c254b5ab34712e101f5ef4892
  Stored in directory: /root/.cache/pip/wheels/7d/95/55/2885348fd3ba98c4badd2ab700ff69f68a73e1098c05959928
Successfully built llama-cpp-python


In [49]:
model_path = "/content/drive/MyDrive/zephyr-7b-beta.Q4_0.gguf"

print(os.path.exists(model_path))

True


In [50]:
from langchain_community.llms import LlamaCpp
llms = LlamaCpp(streaming=True,
                   model_path="/content/drive/MyDrive/zephyr-7b-beta.Q4_0.gguf",
                   max_tokens = 1500,
                   temperature=0.75,
                   top_p=1,
                   gpu_layers=0,
                   stream=True,
                   verbose=True,n_threads = int(os.cpu_count()/2),
                   n_ctx=4096)

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3473: UserWarning: WARNING! gpu_layers is not default parameter.
                gpu_layers was transferred to model_kwargs.
                Please confirm that gpu_layers is what you intended.
  if (await self.run_code(code, result,  async_=asy)):
/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3473: UserWarning: WARNING! stream is not default parameter.
                stream was transferred to model_kwargs.
                Please confirm that stream is what you intended.
  if (await self.run_code(code, result,  async_=asy)):
llama_model_loader: loaded meta data with 21 key-value pairs and 291 tensors from /content/drive/MyDrive/zephyr-7b-beta.Q4_0.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loa

In [52]:
from langchain_classic.chains import RetrievalQA

In [53]:
qa = RetrievalQA.from_chain_type(
      llm=llms,
      chain_type="stuff",
      retriever = compression_retriever_reordered,
      return_source_documents = True
)

In [54]:
query ="who is jon snow?"
results = qa(query)
print(results['result'])
#
print(results["source_documents"])

/tmp/ipykernel_4148/3328378846.py:2: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  results = qa(query)
llama_perf_context_print:        load time =  394137.14 ms
llama_perf_context_print: prompt eval time =  394131.10 ms /  1334 tokens (  295.45 ms per token,     3.38 tokens per second)
llama_perf_context_print:        eval time =  406575.17 ms /   621 runs   (  654.71 ms per token,     1.53 tokens per second)
llama_perf_context_print:       total time =  801720.42 ms /  1955 tokens
llama_perf_context_print:    graphs reused =        778


 Jon Snow is a fictional character from the A Song of Ice and Fire series of fantasy novels by George R. R. Martin, as well as its television adaptation, Game of Thrones. He appears as a major point-of-view character in all six novels published so far and is portrayed by English actor Kit Harington in the television series. Snow starts out as an orphan boy raised within the household of Lord Eddard Stark of Winterfell, where he is trained as a brother of the Night's Watch. The Night's Watch is a sworn order dedicated to protecting the Seven Kingdoms from attacks by wildlings, whom they call "others" or simply "the enemy." Snow becomes the Lord Commander of the Night's Watch and leads them in a battle against the wildling army. He is later revealed to be the legitimate son of the former King Robert Baratheon and Lyanna Stark, who was betrothed to Eddard Stark before her death during childbirth. Jon Snow is now the rightful heir to the Iron Throne.
In A Song of Ice and Fire, many charact

Jon Snow is a fictional character from the A Song of Ice and Fire series of fantasy novels by George R. R. Martin, as well as its television adaptation, Game of Thrones. He appears as a major point-of-view character in all six novels published so far and is portrayed by English actor Kit Harington in the television series. Snow starts out as an orphan boy raised within the household of Lord Eddard Stark of Winterfell, where he is trained as a brother of the Night's Watch.

Harry Potter is the main character in J.K. Rowling's series of novels and films about magic and wizards. He is an orphan who discovers that he has magical powers and goes to attend Hogwarts School of Witchcraft and Wizardry, where he makes friends and battles evil forces like Lord Voldemort.

In [ ]:
results = qa("who is a harry potter?")
print(results['result'])
#
print(results["source_documents"])
#
for source in  results["source_documents"]:
    print(source.metadata)

In [57]:
results = qa.invoke("How does Jon Snow's relationship with the Stark family influence his identity and decisions throughout A Game of Thrones?")


Llama.generate: 197 prefix-match hit, remaining 1129 prompt tokens to eval
llama_perf_context_print:        load time =  394137.14 ms
llama_perf_context_print: prompt eval time =  536840.85 ms /  1601 tokens (  335.32 ms per token,     2.98 tokens per second)
llama_perf_context_print:        eval time =  159768.61 ms /   248 runs   (  644.23 ms per token,     1.55 tokens per second)
llama_perf_context_print:       total time =  500493.84 ms /  1849 tokens
llama_perf_context_print:    graphs reused =        383


In [58]:
results

{'query': "How does Jon Snow's relationship with the Stark family influence his identity and decisions throughout A Game of Thrones?",
 'result': " Jon Snow's relationship with the Stark family plays a significant role in shaping his identity and decision-making throughout A Game of Thrones. As a legitimate son of Eddard Stark, Jon is aware that he belongs to an esteemed noble family. However, due to circumstances beyond his control, Jon chooses to join the Night's Watch instead of inheriting Winterfell and continuing his family's lineage. This decision marks a turning point in Jon's identity formation as it highlights his commitment to duty, honor, and loyalty above personal desires. Furthermore, his interaction with Stark sisters Arya and Sansa also influences his identity as he displays a protective and caring nature towards them, especially when they are both taken captive by different parties. This affection for the Starks reinforces Jon's decision to remain loyal to the Night's W

In [59]:
print(results['result'])



 Jon Snow's relationship with the Stark family plays a significant role in shaping his identity and decision-making throughout A Game of Thrones. As a legitimate son of Eddard Stark, Jon is aware that he belongs to an esteemed noble family. However, due to circumstances beyond his control, Jon chooses to join the Night's Watch instead of inheriting Winterfell and continuing his family's lineage. This decision marks a turning point in Jon's identity formation as it highlights his commitment to duty, honor, and loyalty above personal desires. Furthermore, his interaction with Stark sisters Arya and Sansa also influences his identity as he displays a protective and caring nature towards them, especially when they are both taken captive by different parties. This affection for the Starks reinforces Jon's decision to remain loyal to the Night's Watch as it allows him to continue serving and protecting the realm, while also providing him with opportunities to reunite with his family members 

In [60]:
print(results["source_documents"])



[_DocumentWithState(metadata={'producer': 'pypdf', 'creator': 'PyPDF', 'total_pages': 8, 'source': '/content/got_book.pdf', 'page': 1, 'page_label': '2', 'creationdate': ''}, page_content='raiders. Each day had been worse than the day that had come before it. Today was the \nworst of all. A cold wind was blowing out of the north, and it made the trees rustle like \nliving things. All day, Will had felt as though something were watching him, something \ncold and implacable that loved him not. Gared had felt it too. Will wanted nothing so \nmuch as to ride hellbent for the safety of the Wall, but that was not a feeling to share \nwith your commander.', state={'embedded_doc': [0.007859532721340656, -0.0022724794689565897, -0.022216392681002617, 0.008844316005706787, -0.019976502284407616, -0.025766605511307716, -0.014628252945840359, 0.021754667162895203, 0.009795966558158398, 0.0032267295755445957, 0.04249703139066696, 0.0060594333335757256, 0.01175045594573021, -0.025751637294888496, -0

In [61]:
for source in  results["source_documents"]:
    print(source.metadata)

{'producer': 'pypdf', 'creator': 'PyPDF', 'total_pages': 8, 'source': '/content/got_book.pdf', 'page': 1, 'page_label': '2', 'creationdate': ''}
{'source': '/content/got_book.pdf', 'total_pages': 8, 'producer': 'pypdf', 'creator': 'PyPDF', 'creationdate': '', 'page': 7, 'page_label': '8'}
{'creator': 'PyPDF', 'page_label': '1', 'source': '/content/got_book.pdf', 'creationdate': '', 'page': 0, 'total_pages': 8, 'producer': 'pypdf'}
{'producer': 'pypdf', 'creationdate': '', 'source': '/content/got_book.pdf', 'page': 1, 'creator': 'PyPDF', 'page_label': '2', 'total_pages': 8}
{'page': 2, 'page_label': '3', 'creator': 'PyPDF', 'producer': 'pypdf', 'source': '/content/got_book.pdf', 'total_pages': 8, 'creationdate': ''}
{'gts_pdfxversion': 'PDF/X-4', 'source': '/content/harry_potter_book.pdf', 'creationdate': '2014-01-04T16:40:46+08:00', 'page': 1, 'producer': 'Adobe PDF Library 10.0.1', 'creator': 'Adobe InDesign CS6 (Windows)', 'moddate': '2014-01-04T17:23:49+08:00', 'title': '书籍1.indb', 

In [62]:
for source in results["source_documents"]:
    print(f"Page {source.metadata['page_label']}")

Page 2
Page 8
Page 1
Page 2
Page 3
Page 2
Page 1
Page 5
Page 4
Page 5


## Conclusion


Conclusion

In this implementation, I created a RetrievalQA pipeline using the MergerRetriever combined with LongContextReorder. When the query "Who is Jon Snow?" was asked, the retriever searched across multiple documents, merged the retrieved chunks, reordered them to reduce the Lost in the Middle problem, and passed the optimized context to the LLM.

The LLM generated the final answer using the reordered context, while return_source_documents=True returned the supporting document chunks. By printing the page numbers:

Page 2
Page 8
Page 1
Page 2
Page 3
Page 2
Page 1
Page 5
Page 4
Page 5

I verified that the answer was generated using information retrieved from multiple pages across different PDF documents rather than relying on a single source. This demonstrates that MergerRetriever improves retrieval coverage by combining results from multiple retrievers, and LongContextReorder improves context organization before passing it to the LLM, leading to more accurate and reliable RAG responses for long documents and multi-document knowledge bases.